<a href="https://colab.research.google.com/github/dgaida/rag_foerderkatalog/blob/master/notebooks/RAG_Foerderkatallog_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 RAG Förderkatalog v0.2.0 - Google Colab

[![GitHub Release](https://img.shields.io/badge/release-v0.2.0-blue.svg)](https://github.com/dgaida/rag_foerderkatalog/releases/tag/v0.2.0)
[![Python 3.11+](https://img.shields.io/badge/python-3.11+-blue.svg)](https://www.python.org/downloads/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

**Semantische Suche in deutschen Forschungsförderprojekten**

Dieses Notebook ermöglicht die einfache Nutzung der RAG Förderkatalog-Anwendung in Google Colab:
- ✅ **HuggingFace Embeddings** statt Ollama (Cloud-kompatibel)
- ✅ **Vorbereiteter Index** (~300k Projekte)
- ✅ **Keine lokale Installation** nötig
- ✅ **Gradio Web-UI** im Browser

---

## 📋 Voraussetzungen

- **GROQ API Key** für LLM-Funktionen (kostenlos unter [console.groq.com](https://console.groq.com/))
- **Google Drive** wird temporär für Downloads genutzt (~2GB)
- **Runtime**: Standard-Python (kein GPU nötig)

---

## 🚀 Schritt 1: Installation der Dependencies

In [1]:
%%capture
# Basis-Pakete installieren (dauert ~2-3 Minuten)
!pip install --upgrade pip setuptools wheel

# Core Dependencies
!pip install pandas numpy faiss-cpu gradio python-dotenv tqdm requests

# LLM Client
!pip install git+https://github.com/dgaida/llm_client.git

# HuggingFace Embeddings Support
!pip install llama-index-embeddings-huggingface

print("✅ Dependencies installiert!")

In [2]:
%%capture
# RAG Förderkatalog v0.2.0 installieren
!pip install git+https://github.com/dgaida/rag_foerderkatalog.git@v0.2.0

print("✅ RAG Förderkatalog v0.2.0 installiert!")

## 📥 Schritt 2: Download des vorbereiteten Index

In [ ]:
import os
import zipfile
from pathlib import Path
import requests
from tqdm import tqdm

# Stelle sicher, dass wir in /content sind
os.chdir('/content')

# Erstelle Verzeichnisse mit absoluten Pfaden
COLAB_DATA_DIR = Path('/content/data')
COLAB_INPUT_DIR = Path('/content/input')

COLAB_DATA_DIR.mkdir(parents=True, exist_ok=True)
COLAB_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Download URL
INDEX_URL = "https://github.com/dgaida/rag_foerderkatalog/releases/download/v0.2.0/rag_foerderkatalog_index_v0.2.0.zip"
ZIP_FILE = "/content/rag_index_v0.2.0.zip"

print("📥 Lade vorbereiteten Index herunter...")
print(f"   URL: {INDEX_URL}")
print(f"   Ziel: {ZIP_FILE}")
print(f"   Größe: ~800 MB")
print("")

# Download mit Fortschrittsanzeige
try:
    response = requests.get(INDEX_URL, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))

    with open(ZIP_FILE, 'wb') as f, tqdm(
        desc="Download",
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as pbar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            pbar.update(len(chunk))

    print("")
    print("✅ Download abgeschlossen")

except Exception as e:
    print(f"❌ Download-Fehler: {e}")
    raise

print("")
print("📦 Entpacke Index...")
print(f"   Zielverzeichnis: {COLAB_DATA_DIR}")

# Entpacken direkt nach /content/data/
try:
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        # Liste alle Dateien
        file_list = zip_ref.namelist()
        print(f"   Dateien im Archiv: {len(file_list)}")

        # Entpacke alle Dateien
        for file in tqdm(file_list, desc="Entpacken"):
            zip_ref.extract(file, COLAB_DATA_DIR)

    print("✅ Entpacken abgeschlossen")

except Exception as e:
    print(f"❌ Entpack-Fehler: {e}")
    raise

print("")

# Aufräumen
if Path(ZIP_FILE).exists():
    os.remove(ZIP_FILE)
    print("🧹 ZIP-Datei gelöscht")

print("")
print("📊 Prüfe entpackte Dateien...")

# Prüfe ob Dateien existieren
expected_files = [
    COLAB_DATA_DIR / 'vector_hf.index',
    COLAB_DATA_DIR / 'embeddings_map_hf.json'
]

all_ok = True
for file_path in expected_files:
    exists = file_path.exists()
    status = "✅" if exists else "❌"
    print(f"   {status} {file_path.name}")

    if exists:
        size = file_path.stat().st_size
        if size > 1e9:
            print(f"       Größe: {size / 1e9:.2f} GB")
        else:
            print(f"       Größe: {size / 1e6:.2f} MB")
    else:
        all_ok = False

print("")

if all_ok:
    print("✅ Alle Index-Dateien erfolgreich geladen!")
    print("")
    print("Gefundene Dateien in /content/data/:")
    for item in sorted(COLAB_DATA_DIR.glob('*')):
        print(f"   • {item.name}")
else:
    print("❌ FEHLER: Nicht alle Dateien wurden korrekt entpackt!")
    print("")
    print("Versuchen Sie:")
    print("   1. Diese Zelle erneut ausführen")
    print("   2. Runtime neu starten und von vorne beginnen")

    # Zeige was tatsächlich im Verzeichnis ist
    print("")
    print("Aktueller Inhalt von /content/data/:")
    if COLAB_DATA_DIR.exists():
        for item in COLAB_DATA_DIR.glob('*'):
            print(f"   • {item}")
    else:
        print("   (Verzeichnis existiert nicht)")

## 📄 Schritt 3: CSV-Datei herunterladen

Die CSV-Datei wird vom BMBF Förderkatalog heruntergeladen (~200 MB).

In [ ]:
import requests
from tqdm import tqdm

# BMBF Förderkatalog URL
CSV_URL = "https://foerderportal.bund.de/foekat/export/foerderkatalog_export.csv"
CSV_FILE = "input/foerderkatalog_export.csv"

print("📥 Lade BMBF Förderkatalog CSV...")
print(f"   URL: {CSV_URL}")
print("")

# Download mit Fortschrittsanzeige
response = requests.get(CSV_URL, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(CSV_FILE, 'wb') as f, tqdm(
    desc="Download CSV",
    total=total_size,
    unit='B',
    unit_scale=True,
    unit_divisor=1024,
) as pbar:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        pbar.update(len(chunk))

csv_path = Path(CSV_FILE)
if csv_path.exists():
    print("")
    print(f"✅ CSV erfolgreich heruntergeladen!")
    print(f"   • Datei: {csv_path}")
    print(f"   • Größe: {csv_path.stat().st_size / 1e6:.2f} MB")
else:
    print("❌ Fehler: CSV konnte nicht heruntergeladen werden!")
    raise FileNotFoundError("CSV-Datei fehlt")

## 🚀 Schritt 4: Anwendung starten

Die Gradio-Oberfläche wird automatisch geöffnet. Klicken Sie auf den generierten Link.

In [ ]:
import os
import sys
from pathlib import Path
from src.search.engine import ProjectSearchEngine
from src.app import build_ui
from src.utils.logging_config import setup_logging
import logging

# ===== KRITISCHER COLAB-FIX =====
# In Google Colab müssen wir die Pfade in config.py überschreiben
# da das Working Directory /content/ ist

# Working Directory auf /content setzen (sollte bereits der Fall sein)
os.chdir('/content')

# Pfade für Colab explizit setzen
COLAB_DATA_DIR = Path('/content/data')
COLAB_INPUT_DIR = Path('/content/input')
COLAB_CSV_FILE = COLAB_INPUT_DIR / 'foerderkatalog_export.csv'

# Prüfe ob Dateien existieren
print("🔍 Prüfe Dateisystem...")
print(f"   Working Directory: {os.getcwd()}")
print(f"   Data Dir exists: {COLAB_DATA_DIR.exists()}")
print(f"   CSV exists: {COLAB_CSV_FILE.exists()}")

if COLAB_DATA_DIR.exists():
    print(f"   Files in data/: {list(COLAB_DATA_DIR.glob('*'))}")
else:
    print("   ⚠️ data/ Verzeichnis nicht gefunden!")

print("")

# Setup Logging
setup_logging(level=logging.INFO)

print("🧠 RAG Förderkatalog v0.2.0 (Colab)")
print("="*60)
print("")
print("🔧 Konfiguration:")
print("   • Provider: HuggingFace")
print("   • Modell: intfloat/e5-small-v2")
print("   • Index: Pre-loaded (v0.2.0)")
print("   • Data Dir: /content/data")
print("   • CSV: /content/input/foerderkatalog_export.csv")
print("")

# ===== WICHTIG: Überschreibe config-Pfade für Colab =====
import src.config as config

# Setze alle Pfade auf Colab-spezifische Werte
config.ROOT = Path('/content')
config.INPUT_CSV = COLAB_CSV_FILE
config.DATA_DIR = COLAB_DATA_DIR
config.FAISS_INDEX_FILE_HF = COLAB_DATA_DIR / 'vector_hf.index'
config.EMBED_MAP_FILE_HF = COLAB_DATA_DIR / 'embeddings_map_hf.json'
config.PROGRESS_FILE_HF = COLAB_DATA_DIR / 'indexing_progress_hf.json'
config.LOG_DIR = Path('/content/logs')

# Erstelle Log-Verzeichnis explizit
config.LOG_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Config-Pfade für Colab gesetzt")
print(f"   Log-Verzeichnis erstellt: {config.LOG_DIR}")
print("")

# Prüfe nochmal kritische Dateien
index_file = config.FAISS_INDEX_FILE_HF
map_file = config.EMBED_MAP_FILE_HF

print("📊 Finale Pfadprüfung:")
print(f"   • Index: {index_file}")
print(f"     Exists: {index_file.exists()}")
if index_file.exists():
    print(f"     Size: {index_file.stat().st_size / 1e9:.2f} GB")

print(f"   • Mapping: {map_file}")
print(f"     Exists: {map_file.exists()}")
if map_file.exists():
    print(f"     Size: {map_file.stat().st_size / 1e6:.2f} MB")

print(f"   • CSV: {config.INPUT_CSV}")
print(f"     Exists: {config.INPUT_CSV.exists()}")
if config.INPUT_CSV.exists():
    print(f"     Size: {config.INPUT_CSV.stat().st_size / 1e6:.2f} MB")

print("")

# ===== Engine initialisieren =====
if not index_file.exists() or not map_file.exists():
    print("❌ FEHLER: Index-Dateien nicht gefunden!")
    print("")
    print("Bitte führen Sie Schritt 2 (Download) erneut aus:")
    print("   • Die ZIP-Datei muss nach /content/data/ entpackt werden")
    print("   • Erwartete Dateien:")
    print(f"     - {index_file}")
    print(f"     - {map_file}")
    sys.exit(1)

if not config.INPUT_CSV.exists():
    print("❌ FEHLER: CSV-Datei nicht gefunden!")
    print("")
    print("Bitte führen Sie Schritt 3 (CSV-Download) erneut aus")
    sys.exit(1)

print("⏳ Initialisiere Engine...")

try:
    # Engine mit HuggingFace Provider und Colab-CSV initialisieren
    engine = ProjectSearchEngine(
        provider="huggingface",
        csv_file=config.INPUT_CSV
    )

    # CSV laden
    print("📊 Lade CSV-Daten...")
    engine.load_and_clean()

    # Index-Info anzeigen
    info = engine.get_index_info()
    print("")
    print("📈 Index-Informationen:")
    print(f"   • CSV-Zeilen: {info['csv_rows']:,}")
    print(f"   • Indizierte Vektoren: {info['total_vectors']:,}")
    print(f"   • Embedding-Dimension: {info['dimension']}")

    if info['csv_rows'] > 0 and info['total_vectors'] > 0:
        coverage = (info['total_vectors'] / info['csv_rows']) * 100
        print(f"   • Abdeckung: {coverage:.1f}%")

    print("")
    print("🌐 Starte Gradio-Oberfläche...")
    print("")
    print("👉 Klicken Sie auf den generierten Link unten!")
    print("")

    # Gradio UI starten (mit Live-Logging)
    try:
        from src.app_with_logging import build_ui_with_logging
        print("   🔍 Verwende Debug-Version mit Live-Logging")
        demo = build_ui_with_logging(engine)
    except ImportError:
        print("   ⚠️ Debug-Version nicht verfügbar, verwende Standard-UI")
        demo = build_ui(engine)

    demo.launch(
        share=True,  # Öffentlicher Link (Colab-kompatibel)
        debug=False,
        show_error=True
    )

except Exception as e:
    print(f"❌ FEHLER: {e}")
    import traceback
    traceback.print_exc()
    print("")
    print("Mögliche Lösungen:")
    print("   1. Prüfen Sie, ob alle Dateien korrekt entpackt wurden")
    print("   2. Führen Sie Schritt 2 und 3 erneut aus")
    print("   3. Starten Sie die Runtime neu und versuchen es erneut")

## 💡 Nutzungshinweise

### Suchmodi

- **Hybrid** (empfohlen): Kombiniert semantische und Keyword-Suche
- **Semantic**: Reine KI-basierte Vektorsuche
- **Keyword**: Schnelle textbasierte Suche

### Beispielsuchen

```
Künstliche Intelligenz Hochschule Bayern
Wasserstoff Energie NRW 2020-2025
Quantencomputing Forschung
Klimawandel Digitalisierung
Medizintechnik Berlin
```

### Features

- ✅ **300.000+ Förderprojekte** durchsuchbar
- ✅ **Semantische Suche** findet thematisch ähnliche Projekte
- ✅ **KI-Analyse** generiert Zusammenfassungen
- ✅ **Projekt-Details** per FKZ-Auswahl
- ✅ **Statistiken** zu Fördersummen und Zeiträumen

---

## 🛠️ Fehlerbehebung

### Problem: "Out of Memory"

**Lösung**: Starten Sie die Runtime neu und führen Sie nur die nötigen Zellen aus.

```python
# Runtime neu starten
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)
```

### Problem: "API Key ungültig"

**Lösung**: Überprüfen Sie Ihren GROQ API Key:

```python
import os
print(f"API Key: {os.environ.get('GROQ_API_KEY', 'NICHT GESETZT')}")
```

### Problem: "Index nicht gefunden"

**Lösung**: Führen Sie Schritt 2 (Download) erneut aus.

```python
# Prüfe Index-Dateien
from pathlib import Path
print(f"Index existiert: {Path('data/vector_hf.index').exists()}")
print(f"Mapping existiert: {Path('data/embeddings_map_hf.json').exists()}")
```

### Problem: "HuggingFace Model Download langsam"

**Lösung**: Das erste Laden des Modells dauert 1-2 Minuten. Das ist normal.

---

## ℹ️ Weitere Informationen

### Links

- 📦 [GitHub Repository](https://github.com/dgaida/rag_foerderkatalog)
- 📖 [Dokumentation](https://github.com/dgaida/rag_foerderkatalog#readme)
- 🐛 [Issues](https://github.com/dgaida/rag_foerderkatalog/issues)
- 💬 [Discussions](https://github.com/dgaida/rag_foerderkatalog/discussions)

### Technische Details

- **Python**: 3.11+
- **Embeddings**: HuggingFace (intfloat/e5-small-v2, 384 dim)
- **Vector DB**: FAISS (CPU)
- **LLM**: GROQ API
- **UI**: Gradio 4.0+

### Lizenz

MIT License - siehe [LICENSE](https://github.com/dgaida/rag_foerderkatalog/blob/master/LICENSE)

---

**© 2025 RAG Förderkatalog** | v0.2.0
